# TinyViT Experiment Runner

A minimal notebook to run experiments using your existing config files.

**Two ways to use this:**
1. **Local runtime**: Connect Colab to your local machine (uses your GPU + local files)
2. **Cloud runtime**: Run on Google's GPUs (requires uploading files to Drive)

---

In [ ]:
#@title 1. Environment Check
import torch
import os
import platform

print("="*60)
print("ENVIRONMENT INFO")
print("="*60)
print(f"Platform: {platform.system()} {platform.release()}")
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected!")

# Check if running locally or on Colab
try:
    from google.colab import drive
    IS_COLAB_CLOUD = True
    print("\nRuntime: Google Colab (Cloud)")
    print("-> Your local files are NOT accessible")
    print("-> Using Google's GPU")
except:
    IS_COLAB_CLOUD = False
    print("\nRuntime: Local")
    print("-> Your local files ARE accessible")
    print("-> Using your local GPU")

print(f"\nCurrent directory: {os.getcwd()}")

In [ ]:
#@title 2. Set Working Directory
import os

# Change this to your TinyViT directory
TINYVIT_DIR = r"C:\Users\ilyas\TinyViT\Cream\TinyViT"  #@param {type:"string"}

# For Colab cloud, use Google Drive path instead:
# TINYVIT_DIR = "/content/drive/MyDrive/TinyViT/Cream/TinyViT"

if os.path.exists(TINYVIT_DIR):
    os.chdir(TINYVIT_DIR)
    print(f"Working directory: {os.getcwd()}")
    print(f"\nAvailable config directories:")
    for d in os.listdir('configs'):
        print(f"  - configs/{d}/")
else:
    print(f"ERROR: Directory not found: {TINYVIT_DIR}")
    print("If using Colab cloud, mount Google Drive first.")

In [ ]:
#@title 3. List Available Configs
import os
from pathlib import Path

CONFIG_DIR = "configs/cifar100/experiments"  #@param {type:"string"}

if os.path.exists(CONFIG_DIR):
    configs = sorted(Path(CONFIG_DIR).glob("*.yaml"))
    print(f"Available configs in {CONFIG_DIR}:")
    print("="*60)
    for i, cfg in enumerate(configs):
        print(f"{i+1:2d}. {cfg.name}")
else:
    print(f"Directory not found: {CONFIG_DIR}")

In [ ]:
#@title 4. Configure Experiment

# ============================================================================
# MAIN CONFIGURATION - Edit these values
# ============================================================================

#@markdown ### Config File
CONFIG_FILE = "configs/cifar100/experiments/exp3_teacher_tinyvit21m.yaml"  #@param {type:"string"}

#@markdown ### Paths
DATA_PATH = "./data"  #@param {type:"string"}
OUTPUT_DIR = "./output/exp3_teacher"  #@param {type:"string"}

#@markdown ### Pretrained Weights (leave empty if not finetuning)
PRETRAINED = "pretrained/tiny_vit_21m_22k_distill.pth"  #@param {type:"string"}

#@markdown ### Resume from checkpoint (leave empty to start fresh)
RESUME = ""  #@param {type:"string"}

#@markdown ### Override Options (format: KEY1 VALUE1 KEY2 VALUE2 ...)
OVERRIDE_OPTS = "DATA.BATCH_SIZE 32 DATA.NUM_WORKERS 0 DATA.PIN_MEMORY False"  #@param {type:"string"}

#@markdown ### Evaluation only?
EVAL_ONLY = False  #@param {type:"boolean"}

#@markdown ### Use Weights & Biases?
USE_WANDB = False  #@param {type:"boolean"}
WANDB_RUN_NAME = "experiment"  #@param {type:"string"}

# Verify config exists
if os.path.exists(CONFIG_FILE):
    print(f"Config file: {CONFIG_FILE}")
    # Show config content
    with open(CONFIG_FILE, 'r') as f:
        print("\nConfig preview:")
        print("="*60)
        print(f.read()[:1500] + "..." if len(f.read()) > 1500 else "")
else:
    print(f"ERROR: Config file not found: {CONFIG_FILE}")

In [ ]:
#@title 5. Build & Display Command

# Build command parts
cmd_parts = [
    "python main.py",
    f"--cfg {CONFIG_FILE}",
    f"--data-path {DATA_PATH}",
    f"--output {OUTPUT_DIR}",
]

if PRETRAINED and os.path.exists(PRETRAINED):
    cmd_parts.append(f"--pretrained {PRETRAINED}")
elif PRETRAINED:
    print(f"WARNING: Pretrained file not found: {PRETRAINED}")

if RESUME and os.path.exists(RESUME):
    cmd_parts.append(f"--resume {RESUME}")
elif RESUME:
    print(f"WARNING: Resume checkpoint not found: {RESUME}")

if EVAL_ONLY:
    cmd_parts.append("--eval")

if USE_WANDB:
    cmd_parts.append("--use-wandb")
    cmd_parts.append(f"--wandb-run-name {WANDB_RUN_NAME}")

if OVERRIDE_OPTS:
    cmd_parts.append(f"--opts {OVERRIDE_OPTS}")

TRAIN_CMD = " ".join(cmd_parts)

print("="*60)
print("TRAINING COMMAND")
print("="*60)
print(TRAIN_CMD)
print("="*60)

In [ ]:
#@title 6. Run Training
import os

# Set environment variables for distributed training (single GPU)
os.environ['RANK'] = '0'
os.environ['WORLD_SIZE'] = '1'
os.environ['LOCAL_RANK'] = '0'
os.environ['MASTER_ADDR'] = 'localhost'
os.environ['MASTER_PORT'] = '29500'

print("Starting training...")
print("="*60)

# Run the command
!{TRAIN_CMD}

---
## Quick Experiments

Run one of these cells to quickly configure common experiments, then run cells 5-6.

In [ ]:
#@title Quick: Finetune TinyViT-21M Teacher
CONFIG_FILE = "configs/cifar100/experiments/exp3_teacher_tinyvit21m.yaml"
DATA_PATH = "./data"
OUTPUT_DIR = "./output/exp3_teacher_tinyvit21m"
PRETRAINED = "pretrained/tiny_vit_21m_22k_distill.pth"
RESUME = ""
OVERRIDE_OPTS = "DATA.BATCH_SIZE 32 DATA.NUM_WORKERS 0 DATA.PIN_MEMORY False"
EVAL_ONLY = False
USE_WANDB = False
print("Configured: Finetune TinyViT-21M Teacher")
print("Now run cells 5 and 6")

In [ ]:
#@title Quick: Online Distillation (21M -> 5M) with Features
CONFIG_FILE = "configs/cifar100/experiments/exp10_online_distill_tinyvit21m_to_5m.yaml"
DATA_PATH = "./data"
OUTPUT_DIR = "./output/exp10_online_distill"
PRETRAINED = ""
RESUME = ""
# Set teacher checkpoint path
TEACHER_CKPT = "./output/exp3_teacher_tinyvit21m/Exp3-TinyViT21M-Teacher/default/ckpt_best.pth"
OVERRIDE_OPTS = f"DATA.BATCH_SIZE 16 DATA.NUM_WORKERS 0 DATA.PIN_MEMORY False DISTILL.TEACHER_CHECKPOINT {TEACHER_CKPT}"
EVAL_ONLY = False
USE_WANDB = False
print("Configured: Online Distillation with Features")
print(f"Teacher checkpoint: {TEACHER_CKPT}")
print("Now run cells 5 and 6")

In [ ]:
#@title Quick: Online Distillation (Logits Only)
CONFIG_FILE = "configs/cifar100/experiments/exp10_online_distill_logits_only.yaml"
DATA_PATH = "./data"
OUTPUT_DIR = "./output/exp10_online_logits_only"
PRETRAINED = ""
RESUME = ""
TEACHER_CKPT = "./output/exp3_teacher_tinyvit21m/Exp3-TinyViT21M-Teacher/default/ckpt_best.pth"
OVERRIDE_OPTS = f"DATA.BATCH_SIZE 16 DATA.NUM_WORKERS 0 DATA.PIN_MEMORY False DISTILL.TEACHER_CHECKPOINT {TEACHER_CKPT}"
EVAL_ONLY = False
USE_WANDB = False
print("Configured: Online Distillation (Logits Only)")
print("Now run cells 5 and 6")

In [ ]:
#@title Quick: Saved Logits Distillation (TopK Ablation)
K_VALUE = 50  #@param [10, 25, 50, 100]
CONFIG_FILE = f"configs/cifar100/experiments/exp7_ablation_k{K_VALUE}.yaml"
DATA_PATH = "./data"
OUTPUT_DIR = f"./output/exp7_ablation_k{K_VALUE}"
PRETRAINED = ""
RESUME = ""
LOGITS_PATH = f"./output/logits/tinyvit21m_k{K_VALUE}/"
OVERRIDE_OPTS = f"DATA.BATCH_SIZE 64 DATA.NUM_WORKERS 0 DATA.PIN_MEMORY False DISTILL.TEACHER_LOGITS_PATH {LOGITS_PATH}"
EVAL_ONLY = False
USE_WANDB = False
print(f"Configured: TopK Ablation (K={K_VALUE})")
print(f"Logits path: {LOGITS_PATH}")
print("Now run cells 5 and 6")

---
## Utilities

In [ ]:
#@title List Checkpoints
import glob

SEARCH_DIR = "./output"  #@param {type:"string"}

checkpoints = glob.glob(f"{SEARCH_DIR}/**/ckpt_best.pth", recursive=True)
print(f"Found {len(checkpoints)} best checkpoints:")
print("="*60)
for ckpt in checkpoints:
    print(ckpt)

In [ ]:
#@title Evaluate Checkpoint
import os

EVAL_CONFIG = "configs/cifar100/experiments/exp3_teacher_tinyvit21m.yaml"  #@param {type:"string"}
EVAL_CHECKPOINT = "./output/exp3_teacher_tinyvit21m/Exp3-TinyViT21M-Teacher/default/ckpt_best.pth"  #@param {type:"string"}

os.environ['RANK'] = '0'
os.environ['WORLD_SIZE'] = '1'
os.environ['LOCAL_RANK'] = '0'
os.environ['MASTER_ADDR'] = 'localhost'
os.environ['MASTER_PORT'] = '29500'

eval_cmd = f"python main.py --cfg {EVAL_CONFIG} --data-path ./data --output ./output/eval --resume {EVAL_CHECKPOINT} --eval --opts DATA.NUM_WORKERS 0 DATA.PIN_MEMORY False"
print(f"Running: {eval_cmd}")
!{eval_cmd}

In [ ]:
#@title Download Pretrained Weights
import os

os.makedirs('pretrained', exist_ok=True)

DOWNLOAD_21M = True  #@param {type:"boolean"}
DOWNLOAD_5M = True  #@param {type:"boolean"}

if DOWNLOAD_21M and not os.path.exists('pretrained/tiny_vit_21m_22k_distill.pth'):
    print("Downloading TinyViT-21M...")
    !wget -q -P pretrained/ https://github.com/wkcn/TinyViT-model-zoo/releases/download/checkpoints/tiny_vit_21m_22k_distill.pth
    print("Done!")

if DOWNLOAD_5M and not os.path.exists('pretrained/tiny_vit_5m_22k_distill.pth'):
    print("Downloading TinyViT-5M...")
    !wget -q -P pretrained/ https://github.com/wkcn/TinyViT-model-zoo/releases/download/checkpoints/tiny_vit_5m_22k_distill.pth
    print("Done!")

!ls -la pretrained/